# Per-Muscle and Overall Average Metrics — P* Subjects Only

Identical to `algos_avg_results.ipynb` but **only rows where the subject ID
starts with `P` (e.g. P004, P005 …) are included** in the averages.
HV* subjects are excluded.

Subject identification works for both CSV formats:
- New format: uses the `subject` column directly.
- Old format: extracts the subject from the `pred_label` filename.

In [1]:
import pathlib
import re
import os
import numpy as np
import pandas as pd

EVAL_DIR    = pathlib.Path(r'C:\Projects\dissector\eval_notebooks')
SUMMARY_DIR = EVAL_DIR / 'summary_results_P_only'
SUMMARY_DIR.mkdir(exist_ok=True)

# ── Canonical muscle names ────────────────────────────────────────────────────
MUSCLE_ALIASES = {
    'r_gracilis':  'R_gracilis',
    'l_gracilis':  'L_gracilis',
    'r_sartorius': 'R_sartorius',
    'l_sartorius': 'L_sartorius',
    'r_sart':      'R_sartorius',
    'l_sart':      'L_sartorius',
}

# ── Canonical metric names ────────────────────────────────────────────────────
# inter_slice_dice_* MUST come before dice (they contain the substring "dice")
# dice pattern is anchored (^...$) so it never matches a longer column name
_METRIC_RE = [
    ('inter_slice_dice_pred', re.compile(r'inter_slice_dice_pred',         re.I)),
    ('inter_slice_dice_gt',   re.compile(r'inter_slice_dice_gt',           re.I)),
    ('dice',                  re.compile(r'^(?:lower_)?dice$',             re.I)),
    ('hausdorff',             re.compile(r'hausdorff',                     re.I)),
    ('jaccard',               re.compile(r'jaccard',                       re.I)),
    ('volume_similarity',     re.compile(r'volume_similarity',             re.I)),
    ('false_negative',        re.compile(r'false.?neg|falseNeg',           re.I)),
    ('false_positive',        re.compile(r'false.?pos|falsePo',            re.I)),
    ('bce',                   re.compile(r'\bbce\b|binary_cross_entropy',  re.I)),
    ('boundary_iou_3d',       re.compile(r'boundary_iou',                  re.I)),
]

_MUSCLE_PREFIX_RE      = re.compile(r'^[RrLl]_(?:gracilis|sartorius|sart)_', re.I)
_SUBJECT_FROM_FILE_RE  = re.compile(r'^(.+?)_(WATER|FATFRACTION)_stack', re.I)
_FLOAT64_MAX           = np.finfo(np.float64).max


def canonical_metric(col):
    bare = _MUSCLE_PREFIX_RE.sub('', col).rstrip(':')
    for name, pat in _METRIC_RE:
        if pat.search(bare):
            return name
    return None


def extract_muscle(stem):
    s = stem.lower()
    for alias in sorted(MUSCLE_ALIASES, key=len, reverse=True):
        if f'_{alias}_' in s or s.endswith(f'_{alias}'):
            return MUSCLE_ALIASES[alias]
    return None


def get_subjects(df):
    """Return a Series of subject IDs for each row, handling both CSV formats."""
    if 'subject' in df.columns:
        return df['subject'].astype(str)
    # read_csv(index_col=0) puts 'subject' into the index rather than columns;
    # this happens for all CSVs whose first column is 'subject' (e.g. dixon).
    if df.index.name == 'subject':
        return df.index.astype(str)
    for col in ('pred_label', 'pred_file', 'image'):
        if col in df.columns:
            def _extract(val):
                m = _SUBJECT_FROM_FILE_RE.match(os.path.basename(str(val)))
                return m.group(1) if m else None
            return df[col].apply(_extract).astype(str)
    return pd.Series([''] * len(df), index=df.index)


def process_algorithm(label, results_dir):
    """Return a summary DataFrame for P* subjects only, or None if no CSVs found."""
    results_dir = pathlib.Path(results_dir)
    csv_files   = sorted(results_dir.glob('*.csv'))
    if not csv_files:
        print(f'  [skip] no CSVs in {results_dir}')
        return None

    rows = []
    for csv_path in csv_files:
        muscle = extract_muscle(csv_path.stem)
        if muscle is None:
            print(f'  [skip] could not identify muscle in {csv_path.name}')
            continue

        df = pd.read_csv(csv_path, index_col=0)

        # ── Filter to P* subjects only ────────────────────────────────────
        subjects = get_subjects(df)
        mask     = subjects.str.startswith('P', na=False)
        df       = df[mask]

        if df.empty:
            print(f'  [skip] no P* rows in {csv_path.name}')
            continue

        print(f'  {csv_path.name}: {mask.sum()} P* rows (of {len(mask)} total)')

        metric_vals = {}
        for col in df.columns:
            metric_name = canonical_metric(col)
            if metric_name is None:
                continue

            # ── SAFE NUMERIC CONVERSION ──────────────────────────────────
            s = pd.to_numeric(df[col], errors='coerce').to_numpy(dtype=np.float64)

            # Replace inf/-inf AND float64-max sentinel values (stored overflow)
            # with NaN so nanmean never overflows.
            s = np.where(np.isfinite(s) & (np.abs(s) < _FLOAT64_MAX), s, np.nan)

            finite = np.isfinite(s)
            if not finite.any():
                print(f'  [ALL NON-FINITE] {csv_path.name} | {col}')
                continue

            vals    = s[finite]
            min_val = np.min(vals)
            max_val = np.max(vals)

            if metric_name in {'dice', 'jaccard', 'volume_similarity'}:
                if max_val > 1.0 or min_val < -0.1:
                    print(
                        f'  [SCALE ERROR] {csv_path.name} | {col} '
                        f'| expected ~[0,1], got min={min_val:.3f}, max={max_val:.3f}'
                    )

            if metric_name in {'false_negative', 'false_positive'}:
                if max_val > 1e3:
                    print(
                        f'  [COUNT-LIKE FN/FP] {csv_path.name} | {col} '
                        f'| max={max_val:.3e} (likely unnormalized)'
                    )

            metric_vals[metric_name] = np.nanmean(s, dtype=np.float64)

        # ── DERIVED: inter-slice dice ratio (pred / gt) ──────────────────
        # Normalises prediction smoothness against the object's inherent
        # slice-to-slice variation; 1.0 = as consistent as the GT shape.
        pred = metric_vals.get('inter_slice_dice_pred')
        gt   = metric_vals.get('inter_slice_dice_gt')
        if pred is not None and gt is not None and gt > 0:
            metric_vals['inter_slice_dice_ratio'] = pred / gt

        row = {'muscle': muscle}
        row.update(metric_vals)
        rows.append(row)

    if not rows:
        return None

    summary = pd.DataFrame(rows).set_index('muscle')
    summary.insert(0, 'algorithm', label)

    numeric = summary.select_dtypes(include='number').astype(np.float64)
    overall = numeric.mean().rename('Overall_Mean')
    overall['algorithm'] = label
    summary = pd.concat([summary, overall.to_frame().T])
    summary.index.name = 'muscle'
    return summary


print('Helpers ready.')

Helpers ready.


In [2]:
# ── Algorithm registry ────────────────────────────────────────────────────────
REGISTRY = [
    ('MuscleMap Thigh (water)',           'muscle_map_thigh/results_water'),
    ('MuscleMap Thigh (fat fraction)',    'muscle_map_thigh/results_fat_frac'),
    ('MuscleMap WB (water)',              'muscle_map_wb/results_water'),
    ('MuscleMap WB (fat fraction)',       'muscle_map_wb/results_fat_frac'),
    ('MM WB + MedSAM bbox (water)',       'muscle_map_wb_boxes_medsam/results_water'),
    ('MM WB + MedSAM bbox (fat fraction)','muscle_map_wb_boxes_medsam/results_fatfrac'),
    ('MM WB + MedSAM mask (water)',        'muscle_map_wb_masks_medsam/results_water'),
    ('MM WB + MedSAM mask (fat fraction)', 'muscle_map_wb_masks_medsam/results_fat_frac'),
    ('MM WB + SLM-SAM2 (water)',          'muscle_map_wb+slmsam/results_water'),
    ('MM WB + SLM-SAM2 (fat fraction)',   'muscle_map_wb+slmsam/results_fat_frac'),
    ('Dafne (water)',                     'dafne/results_water'),
    ('Dafne (fat fraction)',              'dafne/results_fat_frac'),
    ('Dafne + MedSAM (water)',            'dafne_and_medsam/results_water'),
    ('Dafne + MedSAM (fat fraction)',     'dafne_and_medsam/results_fat_frac'),
    ('Hirriririir (water)',               'multimodal-multiethnic/results_water'),
    ('Hirriririir (fat fraction)',        'multimodal-multiethnic/results_fat_frac'),
    ('MuSeg (water)',                     'museg/results_water'),
    ('MuSeg (fat fraction)',              'museg/results_fat_frac'),
    ('MuSeg (dixon)',                     'museg/results_dixon'),
    ('MedCLIP-SAMv2 (water)',             'medclipsamv2/results_water'),
    ('MedCLIP-SAMv2 (fat fraction)',      'medclipsamv2/results_fat_frac'),
    ('MedCLIP-SAMv2 + MM Boxes (water)',        'medclipsamv2plusboxes/results_water'),
    ('MedCLIP-SAMv2 + MM Boxes (fat fraction)', 'medclipsamv2plusboxes/results_fat_frac'),
    ('MedCLIP-SAMv2 Text+Boxes (water)',        'medclipsamv2textboxes/results_water'),
    ('MedCLIP-SAMv2 Text+Boxes (fat fraction)', 'medclipsamv2textboxes/results_fat_frac'),
    ('MedSegDiff (both channels)',                'medsegdiff/both_channels'),
    ('MedSegDiff (fat fraction)',         'medsegdiff/results_fat_frac'),
    ('MedSegDiff (water)',                'medsegdiff/results_water'),
]

print(f'{len(REGISTRY)} entries in registry.')
for label, rdir in REGISTRY:
    path = EVAL_DIR / rdir
    n = len(list(path.glob('*.csv'))) if path.exists() else 0
    status = f'{n} CSVs' if path.exists() else 'DIR MISSING'
    print(f'  {label}: {status}')

27 entries in registry.
  MuscleMap Thigh (water): 4 CSVs
  MuscleMap Thigh (fat fraction): 4 CSVs
  MuscleMap WB (water): 4 CSVs
  MuscleMap WB (fat fraction): 4 CSVs
  MM WB + MedSAM bbox (water): 4 CSVs
  MM WB + MedSAM bbox (fat fraction): 4 CSVs
  MM WB + MedSAM mask (water): 4 CSVs
  MM WB + MedSAM mask (fat fraction): 4 CSVs
  MM WB + SLM-SAM2 (water): 4 CSVs
  MM WB + SLM-SAM2 (fat fraction): 4 CSVs
  Dafne (water): 4 CSVs
  Dafne (fat fraction): 4 CSVs
  Dafne + MedSAM (water): 4 CSVs
  Dafne + MedSAM (fat fraction): 4 CSVs
  Hirriririir (water): 4 CSVs
  Hirriririir (fat fraction): 4 CSVs
  MuSeg (water): 4 CSVs
  MuSeg (fat fraction): 4 CSVs
  MuSeg (dixon): 4 CSVs
  MedCLIP-SAMv2 (water): 4 CSVs
  MedCLIP-SAMv2 (fat fraction): 4 CSVs
  MedCLIP-SAMv2 + MM Boxes (water): DIR MISSING
  MedCLIP-SAMv2 + MM Boxes (fat fraction): DIR MISSING
  MedCLIP-SAMv2 Text+Boxes (water): 4 CSVs
  MedCLIP-SAMv2 Text+Boxes (fat fraction): 0 CSVs
  MedSegDiff (water): 1 CSVs
  MedSegDiff (fat f

In [3]:
# ── Process all algorithms (P* subjects only) ─────────────────────────────────
summaries = {}

for label, rdir in REGISTRY:
    print(f'\n── {label} ──')
    df = process_algorithm(label, EVAL_DIR / rdir)
    if df is None:
        continue
    summaries[label] = df

    num_cols = df.select_dtypes(include='number').columns
    display(df.reset_index().style.format('{:.4f}', subset=num_cols).hide(axis='index'))

    safe_name = re.sub(r'[^\w]+', '_', label).strip('_').lower()
    out_path  = SUMMARY_DIR / f'{safe_name}_avg_metrics_P.csv'
    df.to_csv(out_path, float_format='%.4f')
    print(f'  Saved -> {out_path}')

print(f'\nProcessed {len(summaries)}/{len(REGISTRY)} algorithms.')


── MuscleMap Thigh (water) ──
  df_L_gracilis_musclemap_thigh_water.csv: 12 P* rows (of 46 total)
  [SCALE ERROR] df_L_gracilis_musclemap_thigh_water.csv | L_gracilis_volume_similarity | expected ~[0,1], got min=-1.634, max=1.507
  df_L_sartorius_musclemap_thigh_water.csv: 12 P* rows (of 46 total)
  [SCALE ERROR] df_L_sartorius_musclemap_thigh_water.csv | L_sartorius_volume_similarity | expected ~[0,1], got min=-0.557, max=1.861
  df_R_gracilis_musclemap_thigh_water.csv: 12 P* rows (of 46 total)
  [SCALE ERROR] df_R_gracilis_musclemap_thigh_water.csv | R_gracilis_volume_similarity | expected ~[0,1], got min=-0.747, max=1.563
  df_R_sartorius_musclemap_thigh_water.csv: 12 P* rows (of 46 total)
  [SCALE ERROR] df_R_sartorius_musclemap_thigh_water.csv | R_sartorius_volume_similarity | expected ~[0,1], got min=-0.798, max=1.545


muscle,algorithm,dice,hausdorff,jaccard,volume_similarity,false_negative,false_positive,bce,boundary_iou_3d,inter_slice_dice_pred,inter_slice_dice_gt,inter_slice_dice_ratio
L_gracilis,MuscleMap Thigh (water),0.565452,14.148585,0.429414,0.076041,0.348035,0.000483,0.012469,0.377924,0.787706,0.786372,1.001697
L_sartorius,MuscleMap Thigh (water),0.558608,18.561670,0.426414,0.395701,0.249315,0.000678,0.015200,0.354873,0.768127,0.880935,0.871945
R_gracilis,MuscleMap Thigh (water),0.548189,11.504531,0.410036,0.194223,0.389180,0.000398,0.011294,0.365574,0.729096,0.782798,0.931397
R_sartorius,MuscleMap Thigh (water),0.635019,14.009165,0.484009,0.093912,0.297579,0.000368,0.011687,0.414034,0.800807,0.867577,0.923038
Overall_Mean,MuscleMap Thigh (water),0.576817,14.555988,0.437468,0.189969,0.321027,0.000482,0.012663,0.378101,0.771434,0.829420,0.932019


  Saved -> C:\Projects\dissector\eval_notebooks\summary_results_P_only\musclemap_thigh_water_avg_metrics_P.csv

── MuscleMap Thigh (fat fraction) ──
  df_L_gracilis_musclemap_thigh_fatfrac.csv: 12 P* rows (of 54 total)
  [SCALE ERROR] df_L_gracilis_musclemap_thigh_fatfrac.csv | L_gracilis_volume_similarity | expected ~[0,1], got min=-1.716, max=0.991
  df_L_sartorius_musclemap_thigh_fatfrac.csv: 12 P* rows (of 54 total)
  [SCALE ERROR] df_L_sartorius_musclemap_thigh_fatfrac.csv | L_sartorius_volume_similarity | expected ~[0,1], got min=-0.714, max=1.391
  df_R_gracilis_musclemap_thigh_fatfrac.csv: 12 P* rows (of 54 total)
  [SCALE ERROR] df_R_gracilis_musclemap_thigh_fatfrac.csv | R_gracilis_volume_similarity | expected ~[0,1], got min=-1.440, max=0.421
  df_R_sartorius_musclemap_thigh_fatfrac.csv: 12 P* rows (of 54 total)
  [SCALE ERROR] df_R_sartorius_musclemap_thigh_fatfrac.csv | R_sartorius_volume_similarity | expected ~[0,1], got min=-0.877, max=1.237


muscle,algorithm,dice,hausdorff,jaccard,volume_similarity,false_negative,false_positive,bce,boundary_iou_3d,inter_slice_dice_pred,inter_slice_dice_gt,inter_slice_dice_ratio
L_gracilis,MuscleMap Thigh (fat fraction),0.577034,11.850291,0.430108,-0.067529,0.383930,0.000427,0.012366,0.377612,0.808255,0.786372,1.027828
L_sartorius,MuscleMap Thigh (fat fraction),0.672256,13.734242,0.523221,0.014436,0.294874,0.000470,0.013703,0.412460,0.854482,0.880935,0.969972
R_gracilis,MuscleMap Thigh (fat fraction),0.622614,9.765459,0.474368,-0.293001,0.419220,0.000216,0.009707,0.392360,0.791760,0.782798,1.011448
R_sartorius,MuscleMap Thigh (fat fraction),0.689422,10.891593,0.540203,-0.160306,0.337613,0.000226,0.011364,0.428431,0.863807,0.867577,0.995654
Overall_Mean,MuscleMap Thigh (fat fraction),0.640331,11.560396,0.491975,-0.126600,0.358909,0.000335,0.011785,0.402716,0.829576,0.829420,1.001226


  Saved -> C:\Projects\dissector\eval_notebooks\summary_results_P_only\musclemap_thigh_fat_fraction_avg_metrics_P.csv

── MuscleMap WB (water) ──
  df_L_gracilis_musclemap_wb_water.csv: 12 P* rows (of 46 total)
  [SCALE ERROR] df_L_gracilis_musclemap_wb_water.csv | L_gracilis_volume_similarity | expected ~[0,1], got min=-1.846, max=0.199
  df_L_sartorius_musclemap_wb_water.csv: 12 P* rows (of 46 total)
  [SCALE ERROR] df_L_sartorius_musclemap_wb_water.csv | L_sartorius_volume_similarity | expected ~[0,1], got min=-0.672, max=0.260
  df_R_gracilis_musclemap_wb_water.csv: 12 P* rows (of 46 total)
  [SCALE ERROR] df_R_gracilis_musclemap_wb_water.csv | R_gracilis_volume_similarity | expected ~[0,1], got min=-1.783, max=-0.079
  df_R_sartorius_musclemap_wb_water.csv: 12 P* rows (of 46 total)
  [SCALE ERROR] df_R_sartorius_musclemap_wb_water.csv | R_sartorius_volume_similarity | expected ~[0,1], got min=-0.810, max=0.005


...

  Saved -> C:\Projects\dissector\eval_notebooks\summary_results_P_only\medclip_samv2_fat_fraction_avg_metrics_P.csv

── MedCLIP-SAMv2 + MM Boxes (water) ──
  [skip] no CSVs in C:\Projects\dissector\eval_notebooks\medclipsamv2plusboxes\results_water

── MedCLIP-SAMv2 + MM Boxes (fat fraction) ──
  [skip] no CSVs in C:\Projects\dissector\eval_notebooks\medclipsamv2plusboxes\results_fat_frac

── MedCLIP-SAMv2 Text+Boxes (water) ──
  df_L_gracilis_mcsam2textboxes_water.csv: 12 P* rows (of 46 total)
  [SCALE ERROR] df_L_gracilis_mcsam2textboxes_water.csv | L_gracilis_volume_similarity | expected ~[0,1], got min=-2.000, max=-0.559
  df_L_sartorius_mcsam2textboxes_water.csv: 12 P* rows (of 46 total)
  [SCALE ERROR] df_L_sartorius_mcsam2textboxes_water.csv | L_sartorius_volume_similarity | expected ~[0,1], got min=-1.834, max=-0.171
  df_R_gracilis_mcsam2textboxes_water.csv: 12 P* rows (of 46 total)
  [SCALE ERROR] df_R_gracilis_mcsam2textboxes_water.csv | R_gracilis_volume_similarity | expect

muscle,algorithm,dice,hausdorff,jaccard,volume_similarity,false_negative,false_positive,bce,boundary_iou_3d,inter_slice_dice_pred,inter_slice_dice_gt,inter_slice_dice_ratio
L_gracilis,MedCLIP-SAMv2 Text+Boxes (water),0.336832,205.659881,0.246690,-1.314948,0.751151,0.000019,0.475571,0.111753,0.894430,0.786372,1.137414
L_sartorius,MedCLIP-SAMv2 Text+Boxes (water),0.597566,105.241038,0.443324,-0.768466,0.547140,0.000059,0.065468,0.237757,0.921159,0.880935,1.045661
R_gracilis,MedCLIP-SAMv2 Text+Boxes (water),0.348684,191.204150,0.252628,-1.296372,0.746124,0.000008,0.393796,0.116178,0.898695,0.782798,1.148055
R_sartorius,MedCLIP-SAMv2 Text+Boxes (water),0.572794,99.952958,0.418047,-0.841084,0.579766,0.000020,0.043647,0.218706,0.926350,0.867577,1.067744
Overall_Mean,MedCLIP-SAMv2 Text+Boxes (water),0.463969,150.514507,0.340172,-1.055218,0.656045,0.000027,0.244620,0.171099,0.910159,0.829420,1.099718


  Saved -> C:\Projects\dissector\eval_notebooks\summary_results_P_only\medclip_samv2_text_boxes_water_avg_metrics_P.csv

── MedCLIP-SAMv2 Text+Boxes (fat fraction) ──
  [skip] no CSVs in C:\Projects\dissector\eval_notebooks\medclipsamv2textboxes\results_fat_frac

── MedSegDiff (water) ──
  df_R_gracilis_medsegdiff_water.csv: 12 P* rows (of 46 total)
  [SCALE ERROR] df_R_gracilis_medsegdiff_water.csv | R_gracilis_volume_similarity | expected ~[0,1], got min=-2.000, max=-1.959


muscle,algorithm,dice,hausdorff,jaccard,volume_similarity,false_negative,false_positive,bce,boundary_iou_3d,inter_slice_dice_pred,inter_slice_dice_gt,inter_slice_dice_ratio
R_gracilis,MedSegDiff (water),0.004936,396.622401,0.002477,-1.982094,0.997515,0.000521,3.497527,0.001527,0.150549,0.782798,0.192322
Overall_Mean,MedSegDiff (water),0.004936,396.622401,0.002477,-1.982094,0.997515,0.000521,3.497527,0.001527,0.150549,0.782798,0.192322


  Saved -> C:\Projects\dissector\eval_notebooks\summary_results_P_only\medsegdiff_water_avg_metrics_P.csv

── MedSegDiff (fat fraction) ──
  [skip] no CSVs in C:\Projects\dissector\eval_notebooks\medsegdiff\results_fat_frac

Processed 23/27 algorithms.


In [4]:
# ── Combined Overall Means (P* only) ─────────────────────────────────────────
overall_rows = [
    df.loc[['Overall_Mean']]
    for df in summaries.values()
    if 'Overall_Mean' in df.index
]

combined = pd.concat(overall_rows)
combined.index = [row['algorithm'] for _, row in combined.iterrows()]
combined.index.name = 'algorithm'
combined = combined.drop(columns='algorithm')
combined = combined.drop(columns=['inter_slice_dice_pred', 'inter_slice_dice_gt'], errors='ignore')

num_cols = combined.select_dtypes(include='number').columns
display(
    combined.reset_index()
    .style
    .format('{:.4f}', subset=num_cols)
    .hide(axis='index')
    .background_gradient(subset=['dice'], cmap='RdYlGn', axis=0)
    .background_gradient(subset=['hausdorff'], cmap='RdYlGn_r', axis=0)
)

out_combined = SUMMARY_DIR / 'overall_means_P_only.csv'
combined.to_csv(out_combined, float_format='%.4f')
print('Saved ->', out_combined)

algorithm,dice,hausdorff,jaccard,volume_similarity,false_negative,false_positive,bce,boundary_iou_3d,inter_slice_dice_ratio
MuscleMap Thigh (water),0.576817,14.555988,0.437468,0.189969,0.321027,0.000482,0.012663,0.378101,0.932019
MuscleMap Thigh (fat fraction),0.640331,11.560396,0.491975,-0.126600,0.358909,0.000335,0.011785,0.402716,1.001226
MuscleMap WB (water),0.748598,7.127114,0.616825,-0.392049,0.349252,0.000079,0.008537,0.490746,1.081268
MuscleMap WB (fat fraction),0.746353,7.171951,0.613425,-0.425927,0.361251,0.000073,0.008827,0.484607,1.084110
MM WB + MedSAM bbox (water),0.626348,8.657510,0.468368,-0.695730,0.520081,0.000048,0.016655,0.269400,1.093022
MM WB + MedSAM bbox (fat fraction),0.614151,10.227846,0.455400,-0.709771,0.530233,0.000066,0.017447,0.277116,1.064910
MM WB + MedSAM mask (water),0.347455,63.447581,0.229833,-0.749338,0.697711,0.000562,0.056118,0.168639,1.033594
MM WB + MedSAM mask (fat fraction),0.357078,68.165952,0.237133,-0.733796,0.687589,0.000564,0.053318,0.165114,1.008295
MM WB + SLM-SAM2 (water),0.457482,43.286339,0.321005,-0.052239,0.500854,0.000520,0.020423,0.318381,0.813405
MM WB + SLM-SAM2 (fat fraction),0.552361,21.849398,0.404811,-0.385611,0.517551,0.000258,0.016595,0.306777,0.919843


Saved -> C:\Projects\dissector\eval_notebooks\summary_results_P_only\overall_means_P_only.csv
